# BaIP: 5-fold cross-validation on Colab

Runtime > Change runtime type > **L4 GPU** (or A100). The free T4 is too slow for full runs.

One-time setup on Google Drive:
```
MyDrive/BaIP/data/Images/            S###_k.jpg
MyDrive/BaIP/data/Images-2/          S###_k.jpg (the 79 extra files)
MyDrive/BaIP/data/labels20260217_sort3d_hsi_measurements.xlsx
```
Each experiment writes to `MyDrive/BaIP/results/<experiment>/fold0..fold4/` and, once all folds are done,
the pooled result to `cv_summary.txt` and `cv_metrics.json` in the same folder.

**If Colab disconnects:** reconnect and run all cells again. Finished folds are skipped; the fold that was
interrupted starts again from epoch 1.

In [ ]:
BRANCH = 'baip-pipeline'   # git branch to run
EXPERIMENTS = ['resize448_cv', 'crop448_cv']   # 5-fold CV experiments; also: crop1080_cv (A100 only)
FOLDS = []                 # [] = all 5; e.g. [0, 1] to split the work over sessions
OVERRIDES = []             # e.g. ['train.lr=1e-4', 'data.augment=true']
                           # free T4 GPU: add 'train.amp=false' (the T4 has no bf16; fp32 is slower)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
REPO = '/content/repo'
if not os.path.isdir(REPO):
    !git clone -q https://github.com/merijnmolenaar/Textile_Classification_CNN_Merijn.git {REPO}
!cd {REPO} && git fetch -q && git checkout -q {BRANCH} && git pull -q && git log -1 --format='commit %h  %s'
!pip -q install pyyaml openpyxl

In [ ]:
# Check the data is where configs/paths_colab.yaml expects it
for p in ['/content/drive/MyDrive/BaIP/data/Images', '/content/drive/MyDrive/BaIP/data/Images-2']:
    print(p, len(os.listdir(p)) if os.path.isdir(p) else 'MISSING')
print('labels:', os.path.exists('/content/drive/MyDrive/BaIP/data/labels20260217_sort3d_hsi_measurements.xlsx'))

**Smoke test** (a few minutes; the first time also copies the images to local disk): one fold, 1 epoch of 3 batches, written to `results/smoke`. Skip once it has worked.

In [ ]:
!cd {REPO} && python scripts/run_cv.py configs/{EXPERIMENTS[0]}.yaml --paths configs/paths_colab.yaml --folds 0 \
    --set train.epochs=1 --set train.max_batches=3 --set out_dir=/content/drive/MyDrive/BaIP/results/smoke

**Optional: reproduce Elsherif** (one run, his 80/20 split, no CV). Expect roughly 74.6% image level and 76.2% sample level on his validation set. A check that the port works, not a benchmark.

In [ ]:
!cd {REPO} && python scripts/train.py configs/repro_elsherif.yaml --paths configs/paths_colab.yaml

**Full runs.** The first epoch of each fold prints an estimate of the time for that fold.

In [ ]:
ov = ' '.join(f'--set {o}' for o in OVERRIDES)
fl = ('--folds ' + ' '.join(map(str, FOLDS))) if FOLDS else ''
for exp in EXPERIMENTS:
    !cd {REPO} && python scripts/run_cv.py configs/{exp}.yaml --paths configs/paths_colab.yaml {fl} {ov}

In [ ]:
import pandas as pd
r = pd.read_csv('/content/drive/MyDrive/BaIP/results/runs.csv')
r[r['fold'].astype(str) == 'pooled']